In [1]:
# ============================================================
# STEP 13A — PM2.5 causal-distance geometry audit
#
# causal weights: Train-only GCCM
#
# feature standardization:
#   fit Calibration ONLY
#   transform Validation
#
# Compare:
#   ordinary standardized Euclidean distance
#   GCCM-weighted causal distance
#
# Frozen Test untouched
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
from scipy.stats import spearmanr

print("=" * 80)
print("STEP 13A — PM2.5 CAUSAL DISTANCE GEOMETRY AUDIT")
print("=" * 80)

# ============================================================
# 0. Paths
# ============================================================

ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

OUTPUT_DIR = (
    ROOT
    / "GCCP_PM2.5"
    / "outputs"
)

GCCM_OUTPUT_DIR = (
    OUTPUT_DIR
    / "gccm"
)

WEIGHT_PATH = (
    GCCM_OUTPUT_DIR
    / "pm25_gccm_causal_weights_v1.csv"
)

SPLIT_PATH = (
    OUTPUT_DIR
    / "pm25_formal_split_assignment_v1.csv"
)

DATA_PATH = (
    ROOT
    / "GCCP_PM2.5"
    / "00_data_audit"
    / "china_sites_2018_05_PM2.5_all.csv"
)

# ============================================================
# 1. Read frozen causal weights
# ============================================================

weight_df = pd.read_csv(
    WEIGHT_PATH,
    encoding="utf-8-sig"
)

FEATURES = [
    "dem",
    "sp",
    "tp",
    "blh",
    "r",
    "aod_sat",
    "ndvi",
]

weight_map = (
    weight_df
    .set_index("feature")
    ["causal_weight"]
    .to_dict()
)

gamma_map = (
    weight_df
    .set_index("feature")
    ["gamma"]
    .to_dict()
)

gamma = np.array(
    [
        gamma_map[f]
        for f in FEATURES
    ],
    dtype=float
)

causal_weight = np.array(
    [
        weight_map[f]
        for f in FEATURES
    ],
    dtype=float
)

print("\nFrozen GCCM weights:")

weight_check = pd.DataFrame({
    "feature": FEATURES,
    "causal_weight": causal_weight,
    "gamma": gamma,
})

display(
    weight_check
)

print(
    "weight sum =",
    causal_weight.sum()
)

print(
    "gamma mean =",
    gamma.mean()
)

assert np.isclose(
    causal_weight.sum(),
    1.0
)

assert np.isclose(
    gamma.mean(),
    1.0
)

# ============================================================
# 2. Reload frozen formal split
# ============================================================

data = pd.read_csv(
    DATA_PATH,
    encoding="gbk"
)

data = data.rename(
    columns={
        "监测点编码": "id",
        "监测点名称": "name",
        "城市": "city",
        "经度": "lon",
        "纬度": "lat",
    }
)

split_df = pd.read_csv(
    SPLIT_PATH,
    encoding="utf-8-sig"
)

idx_calib = (
    split_df.loc[
        split_df["split"] == "calibration",
        "original_index"
    ]
    .to_numpy(dtype=int)
)

idx_val = (
    split_df.loc[
        split_df["split"] == "validation",
        "original_index"
    ]
    .to_numpy(dtype=int)
)

print(
    "\nCalibration:",
    len(idx_calib)
)

print(
    "Validation :",
    len(idx_val)
)

assert len(idx_calib) == 141
assert len(idx_val) == 142

# ============================================================
# 3. Standardization
#
# SAME convention as Additive-EUC:
#
# StandardScaler fit on Calibration only.
# ============================================================

X_calib_raw = (
    data.iloc[idx_calib]
    [FEATURES]
    .to_numpy(dtype=float)
)

X_val_raw = (
    data.iloc[idx_val]
    [FEATURES]
    .to_numpy(dtype=float)
)

scaler = StandardScaler()

Z_calib = scaler.fit_transform(
    X_calib_raw
)

Z_val = scaler.transform(
    X_val_raw
)

print(
    "\nZ_calib shape:",
    Z_calib.shape
)

print(
    "Z_val shape  :",
    Z_val.shape
)

# ============================================================
# 4. Ordinary standardized Euclidean distance
# ============================================================

D_euc = pairwise_distances(
    Z_val,
    Z_calib,
    metric="euclidean"
)

# ============================================================
# 5. GCCM-weighted causal distance
#
# d_causal =
# sqrt(sum_k gamma_k * (z_i-z_j)^2)
# ============================================================

diff = (
    Z_val[:, None, :]
    -
    Z_calib[None, :, :]
)

D_causal = np.sqrt(
    np.sum(
        gamma[
            None,
            None,
            :
        ]
        * diff**2,
        axis=2
    )
)

assert D_causal.shape == D_euc.shape

# ============================================================
# 6. Equal-weight degeneration check
#
# gamma_k = 1 for all k
# must exactly recover ordinary EUC.
# ============================================================

gamma_equal = np.ones(
    len(FEATURES)
)

D_equal = np.sqrt(
    np.sum(
        gamma_equal[
            None,
            None,
            :
        ]
        * diff**2,
        axis=2
    )
)

max_equal_diff = np.max(
    np.abs(
        D_equal
        - D_euc
    )
)

print("\n" + "=" * 80)
print("DEGENERATION CHECK")
print("=" * 80)

print(
    "max |D_equal - D_EUC| =",
    max_equal_diff
)

assert np.allclose(
    D_equal,
    D_euc,
    atol=1e-12
)

print(
    "✅ Equal causal weights exactly recover standardized EUC"
)

# ============================================================
# 7. Distance distributions
# ============================================================

def dist_summary(x):

    x = np.asarray(
        x,
        dtype=float
    ).ravel()

    return {
        "min":
            np.min(x),

        "p05":
            np.quantile(
                x,
                0.05
            ),

        "p25":
            np.quantile(
                x,
                0.25
            ),

        "median":
            np.median(x),

        "p75":
            np.quantile(
                x,
                0.75
            ),

        "p95":
            np.quantile(
                x,
                0.95
            ),

        "max":
            np.max(x),
    }


distance_summary = pd.DataFrame({
    "ordinary_EUC":
        dist_summary(
            D_euc
        ),

    "causal_weighted":
        dist_summary(
            D_causal
        ),
}).T

print("\n" + "=" * 80)
print("DISTANCE DISTRIBUTION")
print("=" * 80)

display(
    distance_summary
)

# ============================================================
# 8. Global distance correlation
# ============================================================

euc_flat = D_euc.ravel()
causal_flat = D_causal.ravel()

pearson_distance = np.corrcoef(
    euc_flat,
    causal_flat
)[0, 1]

spearman_distance, _ = (
    spearmanr(
        euc_flat,
        causal_flat
    )
)

print("\n" + "=" * 80)
print("DISTANCE CORRELATION")
print("=" * 80)

print(
    "Pearson  =",
    pearson_distance
)

print(
    "Spearman =",
    spearman_distance
)

# ============================================================
# 9. Nearest-neighbor rank changes
#
# For every Validation point,
# compare top-k Calibration neighbors.
# ============================================================

K_LIST = [
    1,
    5,
    10,
    20,
]

neighbor_rows = []

euc_order = np.argsort(
    D_euc,
    axis=1
)

causal_order = np.argsort(
    D_causal,
    axis=1
)

for k in K_LIST:

    overlaps = []

    for i in range(
        len(idx_val)
    ):

        set_euc = set(
            euc_order[
                i,
                :k
            ]
        )

        set_causal = set(
            causal_order[
                i,
                :k
            ]
        )

        overlap = (
            len(
                set_euc
                &
                set_causal
            )
            / k
        )

        overlaps.append(
            overlap
        )

    overlaps = np.asarray(
        overlaps,
        dtype=float
    )

    neighbor_rows.append({
        "k":
            k,

        "mean_overlap":
            overlaps.mean(),

        "median_overlap":
            np.median(
                overlaps
            ),

        "p10_overlap":
            np.quantile(
                overlaps,
                0.10
            ),

        "min_overlap":
            overlaps.min(),

        "fraction_exact_same":
            np.mean(
                overlaps == 1.0
            ),
    })


neighbor_overlap_df = pd.DataFrame(
    neighbor_rows
)

print("\n" + "=" * 80)
print("NEAREST-NEIGHBOR OVERLAP")
print("=" * 80)

display(
    neighbor_overlap_df
)

# ============================================================
# 10. Per-validation-point distance rank correlation
# ============================================================

rank_corr = []

for i in range(
    len(idx_val)
):

    rho_i, _ = spearmanr(
        D_euc[i],
        D_causal[i]
    )

    rank_corr.append(
        rho_i
    )

rank_corr = np.asarray(
    rank_corr,
    dtype=float
)

print("\n" + "=" * 80)
print("PER-POINT RANK CORRELATION")
print("=" * 80)

print(
    pd.Series(
        rank_corr
    ).describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

# ============================================================
# 11. Ratio causal / EUC
# ============================================================

ratio = (
    D_causal
    /
    np.maximum(
        D_euc,
        1e-12
    )
)

print("\n" + "=" * 80)
print("CAUSAL / EUC DISTANCE RATIO")
print("=" * 80)

print(
    pd.Series(
        ratio.ravel()
    ).describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

# ============================================================
# 12. Save
# ============================================================

GCCP_OUTPUT_DIR = (
    OUTPUT_DIR
    / "gccp"
)

GCCP_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

weight_check.to_csv(
    GCCP_OUTPUT_DIR
    / "pm25_gccp_causal_weight_check_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

distance_summary.to_csv(
    GCCP_OUTPUT_DIR
    / "pm25_gccp_distance_summary_v1.csv",
    encoding="utf-8-sig"
)

neighbor_overlap_df.to_csv(
    GCCP_OUTPUT_DIR
    / "pm25_gccp_neighbor_overlap_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame({
    "validation_original_index":
        idx_val,

    "rank_correlation":
        rank_corr,
}).to_csv(
    GCCP_OUTPUT_DIR
    / "pm25_gccp_per_validation_rank_correlation_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ STEP 13A 完成")
print("✅ causal distance geometry 已审计")
print("⚠️ Frozen Test y 仍未读取")

STEP 13A — PM2.5 CAUSAL DISTANCE GEOMETRY AUDIT

Frozen GCCM weights:


,feature,causal_weight,gamma
0,dem,0.143548,1.004839
1,sp,0.155176,1.086235
2,tp,0.105341,0.737385
3,blh,0.124493,0.871452
4,r,0.135885,0.951196
5,aod_sat,0.287181,2.010264
6,ndvi,0.048376,0.338629


weight sum = 0.9999999999999997
gamma mean = 0.9999999999999999

Calibration: 141
Validation : 142

Z_calib shape: (141, 7)
Z_val shape  : (142, 7)

DEGENERATION CHECK
max |D_equal - D_EUC| = 5.2735593669694936e-15
✅ Equal causal weights exactly recover standardized EUC

DISTANCE DISTRIBUTION


,min,p05,p25,median,p75,p95,max
ordinary_EUC,0.044035,1.119863,2.101337,2.981901,4.180837,6.896668,20.456417
causal_weighted,0.042988,1.063835,2.042507,2.880114,4.136175,7.028438,19.242001



DISTANCE CORRELATION
Pearson  = 0.9796908898509745
Spearman = 0.9574058338018675

NEAREST-NEIGHBOR OVERLAP


,k,mean_overlap,median_overlap,p10_overlap,min_overlap,fraction_exact_same
0,1,0.725352,1.00,0.0,0.00,0.725352
1,5,0.759155,0.80,0.6,0.40,0.204225
2,10,0.797183,0.80,0.7,0.50,0.070423
3,20,0.832746,0.85,0.7,0.55,0.021127



PER-POINT RANK CORRELATION
count    142.000000
mean       0.935634
std        0.034251
min        0.823815
10%        0.888277
25%        0.924933
50%        0.942515
75%        0.958084
90%        0.970545
max        0.987692
dtype: float64

CAUSAL / EUC DISTANCE RATIO
count    20022.000000
mean         0.987758
std          0.137193
min          0.582275
1%           0.644440
5%           0.745790
25%          0.907729
50%          0.987121
75%          1.068176
95%          1.225429
99%          1.329448
max          1.409332
dtype: float64

✅ STEP 13A 完成
✅ causal distance geometry 已审计
⚠️ Frozen Test y 仍未读取


In [4]:
# ============================================================
# STEP 13B — PM2.5 GCCP-B / Validation tuning
#
# Controlled comparison:
#
# Additive-EUC:
#   K_geo + K_EUC
#
# GCCP-B:
#   K_geo + K_causal
#
# Same:
#   - Formal split
#   - Random Forest predictor
#   - Calibration residual
#   - Haversine geographic distance
#   - h_geo grid
#   - lambda grid
#   - weighted quantile rule
#   - Validation selection rule
#
# Frozen Test remains untouched.
# ============================================================

from pathlib import Path
import math
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics.pairwise import haversine_distances

print("=" * 80)
print("STEP 13B — PM2.5 GCCP-B / VALIDATION")
print("=" * 80)

# ============================================================
# 0. Paths
# ============================================================

ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

OUTPUT_DIR = (
    ROOT
    / "GCCP_PM2.5"
    / "outputs"
)

GCCP_OUTPUT_DIR = (
    OUTPUT_DIR
    / "gccp"
)

GCCP_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATA_PATH = (
    ROOT
    / "GCCP_PM2.5"
    / "00_data_audit"
    / "china_sites_2018_05_PM2.5_all.csv"
)

SPLIT_PATH = (
    OUTPUT_DIR
    / "pm25_formal_split_assignment_v1.csv"
)

H_GEO_PATH = (
    OUTPUT_DIR
    / "pm25_formal_h_geo_grid_v1.csv"
)

LAMBDA_PATH = (
    OUTPUT_DIR
    / "pm25_formal_lambda_grid_v1.csv"
)

ADDITIVE_PATH = (
    OUTPUT_DIR
    / "pm25_formal_additive_euc_validation_best_v3.csv"
)

# ============================================================
# 1. Load data / split
# ============================================================

data = pd.read_csv(
    DATA_PATH,
    encoding="gbk"
)

data = data.rename(
    columns={
        "监测点编码": "id",
        "监测点名称": "name",
        "城市": "city",
        "经度": "lon",
        "纬度": "lat",
    }
)

split_df = pd.read_csv(
    SPLIT_PATH,
    encoding="utf-8-sig"
)

idx_train = (
    split_df.loc[
        split_df["split"] == "train",
        "original_index"
    ]
    .to_numpy(dtype=int)
)

idx_calib = (
    split_df.loc[
        split_df["split"] == "calibration",
        "original_index"
    ]
    .to_numpy(dtype=int)
)

idx_val = (
    split_df.loc[
        split_df["split"] == "validation",
        "original_index"
    ]
    .to_numpy(dtype=int)
)

assert len(idx_train) == 990
assert len(idx_calib) == 141
assert len(idx_val) == 142

# ============================================================
# 2. Formal predictor
#
# Hyperparameters already selected using Formal Train only.
# Do not tune again.
# ============================================================

FEATURES = [
    "dem",
    "sp",
    "tp",
    "blh",
    "r",
    "aod_sat",
    "ndvi",
]

MODEL_FEATURES = (
    FEATURES
    + ["lon", "lat"]
)

X_train = (
    data.iloc[idx_train]
    [MODEL_FEATURES]
)

X_calib = (
    data.iloc[idx_calib]
    [MODEL_FEATURES]
)

X_val = (
    data.iloc[idx_val]
    [MODEL_FEATURES]
)

y_train = (
    data.iloc[idx_train]
    ["PM2.5"]
    .to_numpy(dtype=float)
)

y_calib = (
    data.iloc[idx_calib]
    ["PM2.5"]
    .to_numpy(dtype=float)
)

y_val = (
    data.iloc[idx_val]
    ["PM2.5"]
    .to_numpy(dtype=float)
)

formal_rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

formal_rf.fit(
    X_train,
    y_train
)

pred_calib = formal_rf.predict(
    X_calib
)

pred_val = formal_rf.predict(
    X_val
)

calib_residual = np.abs(
    y_calib
    - pred_calib
)

print("Train       :", len(idx_train))
print("Calibration :", len(idx_calib))
print("Validation  :", len(idx_val))

# ============================================================
# 3. Harmonized conformal quantile level
# ============================================================

ALPHA = 0.1
TARGET_COVERAGE = 0.90

n_calib = len(
    calib_residual
)

k_local = math.ceil(
    (n_calib + 1)
    * (1 - ALPHA)
)

k_local = min(
    k_local,
    n_calib
)

TAU_LOCAL = (
    k_local
    / n_calib
)

print(
    "\nConformal k:",
    k_local
)

print(
    "tau_local:",
    TAU_LOCAL
)

# ============================================================
# 4. Geographic distance: Haversine km
# ============================================================

EARTH_RADIUS_KM = 6371.0088

coord_calib_rad = np.radians(
    data.iloc[idx_calib]
    [["lat", "lon"]]
    .to_numpy(dtype=float)
)

coord_val_rad = np.radians(
    data.iloc[idx_val]
    [["lat", "lon"]]
    .to_numpy(dtype=float)
)

D_geo = (
    haversine_distances(
        coord_val_rad,
        coord_calib_rad
    )
    * EARTH_RADIUS_KM
)

assert D_geo.shape == (
    142,
    141
)

# ============================================================
# 5. Causal distance
#
# D_causal has already been constructed in STEP 13A.
# Verify shape.
# ============================================================

if "D_causal" not in globals():
    raise RuntimeError(
        "D_causal 不存在。请先运行 STEP 13A。"
    )

assert D_causal.shape == (
    142,
    141
)

# ============================================================
# 6. SAME geographic bandwidth grid as Additive-EUC
# ============================================================

h_geo_df = pd.read_csv(
    H_GEO_PATH,
    encoding="utf-8-sig"
)

H_GEO_GRID_GCCP = (
    h_geo_df[
        "h_geo_km"
    ]
    .to_numpy(dtype=float)
)

# ============================================================
# 7. SAME lambda grid as Additive-EUC
# ============================================================

lambda_df = pd.read_csv(
    LAMBDA_PATH,
    encoding="utf-8-sig"
)

LAMBDA_GRID_GCCP = (
    lambda_df[
        "lambda"
    ]
    .to_numpy(dtype=float)
)

print(
    "\nh_geo candidates:",
    len(H_GEO_GRID_GCCP)
)

print(
    "lambda grid:",
    LAMBDA_GRID_GCCP
)

# ============================================================
# 8. Causal bandwidth grid
#
# SAME construction rule as Additive-EUC feature bandwidth:
#
# lower =
#   max(0.05,
#       0.5 * 5th percentile nearest causal distance)
#
# upper =
#   90th percentile of all causal distances
#
# 20 geometric candidates.
#
# No Validation y is used.
# ============================================================

nearest_causal = (
    D_causal.min(
        axis=1
    )
)

all_causal = (
    D_causal.ravel()
)

h_causal_min = max(
    0.05,
    0.5 * np.quantile(
        nearest_causal,
        0.05
    )
)

h_causal_max = np.quantile(
    all_causal,
    0.90
)

H_CAUSAL_GRID = np.geomspace(
    h_causal_min,
    h_causal_max,
    20
)

print("\n" + "=" * 80)
print("FROZEN GCCP-B PARAMETER GRIDS")
print("=" * 80)

print(
    "h_geo range:",
    H_GEO_GRID_GCCP.min(),
    "→",
    H_GEO_GRID_GCCP.max()
)

print(
    "h_causal range:",
    h_causal_min,
    "→",
    h_causal_max
)

print(
    "h_causal candidates:",
    len(H_CAUSAL_GRID)
)

print(
    "lambda:",
    LAMBDA_GRID_GCCP
)

# ============================================================
# 9. Actual Gaussian kernel
#
# Do NOT independently max-shift kernels,
# because that would change lambda interpretation.
# ============================================================

def raw_gaussian_kernel(
    D,
    h
):
    log_w = (
        -0.5
        * (D / h) ** 2
    )

    log_w = np.clip(
        log_w,
        -700,
        0
    )

    return np.exp(
        log_w
    )

# ============================================================
# 10. Stable weighted quantile
# ============================================================

residuals = np.asarray(
    calib_residual,
    dtype=float
)

resid_order = np.argsort(
    residuals
)

resid_sorted = (
    residuals[
        resid_order
    ]
)

def weighted_q_matrix(
    W,
    tau=TAU_LOCAL
):
    W = np.asarray(
        W,
        dtype=float
    )

    W_sorted = (
        W[
            :,
            resid_order
        ]
    )

    row_sum = (
        W_sorted.sum(
            axis=1,
            keepdims=True
        )
    )

    if np.any(
        (~np.isfinite(row_sum))
        |
        (row_sum <= 0)
    ):
        raise RuntimeError(
            "Invalid kernel weights"
        )

    cum = (
        np.cumsum(
            W_sorted,
            axis=1
        )
        / row_sum
    )

    first_idx = np.argmax(
        cum >= tau,
        axis=1
    )

    return (
        resid_sorted[
            first_idx
        ]
    )

# ============================================================
# 11. Stable n_eff
# ============================================================

def neff_matrix_stable(
    W
):
    W = np.asarray(
        W,
        dtype=float
    )

    row_max = np.max(
        W,
        axis=1,
        keepdims=True
    )

    if np.any(
        (~np.isfinite(row_max))
        |
        (row_max <= 0)
    ):
        raise RuntimeError(
            "Invalid kernel weights"
        )

    W_scaled = (
        W / row_max
    )

    return (
        W_scaled.sum(
            axis=1
        ) ** 2
        /
        (
            W_scaled ** 2
        ).sum(
            axis=1
        )
    )

# ============================================================
# 12. Interval Score
# ============================================================

def interval_score(
    y_true,
    lower,
    upper,
    alpha=0.1
):
    y_true = np.asarray(
        y_true,
        dtype=float
    )

    lower = np.asarray(
        lower,
        dtype=float
    )

    upper = np.asarray(
        upper,
        dtype=float
    )

    width = (
        upper - lower
    )

    below = (
        (lower - y_true)
        * (y_true < lower)
    )

    above = (
        (y_true - upper)
        * (y_true > upper)
    )

    return (
        width
        + (2 / alpha) * below
        + (2 / alpha) * above
    )

# ============================================================
# 13. Precompute kernels
# ============================================================

geo_kernels = {
    float(h):
        raw_gaussian_kernel(
            D_geo,
            h
        )
    for h in H_GEO_GRID_GCCP
}

causal_kernels = {
    float(h):
        raw_gaussian_kernel(
            D_causal,
            h
        )
    for h in H_CAUSAL_GRID
}

print(
    "\n✅ Geographic and causal kernels precomputed"
)

# ============================================================
# 14. Grid search
#
# 25 x 20 x 11 = 5500 combinations
# ============================================================

rows = []

for h_geo in H_GEO_GRID_GCCP:

    K_geo = (
        geo_kernels[
            float(h_geo)
        ]
    )

    for h_causal in H_CAUSAL_GRID:

        K_causal = (
            causal_kernels[
                float(h_causal)
            ]
        )

        for lam in LAMBDA_GRID_GCCP:

            W = (
                lam * K_geo
                +
                (1.0 - lam)
                * K_causal
            )

            q_local = (
                weighted_q_matrix(
                    W
                )
            )

            neff = (
                neff_matrix_stable(
                    W
                )
            )

            lower = (
                pred_val
                - q_local
            )

            upper = (
                pred_val
                + q_local
            )

            covered = (
                (y_val >= lower)
                &
                (y_val <= upper)
            )

            width = (
                upper - lower
            )

            score = (
                interval_score(
                    y_val,
                    lower,
                    upper,
                    alpha=ALPHA
                )
            )

            rows.append({
                "h_geo_km":
                    float(h_geo),

                "h_causal":
                    float(h_causal),

                "lambda":
                    float(lam),

                "coverage":
                    float(
                        np.mean(
                            covered
                        )
                    ),

                "mean_half_width":
                    float(
                        np.mean(
                            q_local
                        )
                    ),

                "mean_width":
                    float(
                        np.mean(
                            width
                        )
                    ),

                "interval_score":
                    float(
                        np.mean(
                            score
                        )
                    ),

                "median_n_eff":
                    float(
                        np.median(
                            neff
                        )
                    ),

                "p10_n_eff":
                    float(
                        np.quantile(
                            neff,
                            0.10
                        )
                    ),

                "p90_n_eff":
                    float(
                        np.quantile(
                            neff,
                            0.90
                        )
                    ),
            })

gccp_grid_df = pd.DataFrame(
    rows
)

print(
    "\nGrid combinations:",
    len(gccp_grid_df)
)

# ============================================================
# 15. SAME frozen selection rule
# ============================================================

feasible = (
    gccp_grid_df[
        gccp_grid_df[
            "coverage"
        ]
        >= TARGET_COVERAGE
    ]
)

if len(feasible) > 0:

    best_idx = (
        feasible[
            "interval_score"
        ]
        .idxmin()
    )

    gccp_feasible = True

    gccp_selection_rule = (
        "coverage>=0.90_then_min_interval_score"
    )

else:

    best_idx = (
        gccp_grid_df[
            "interval_score"
        ]
        .idxmin()
    )

    gccp_feasible = False

    gccp_selection_rule = (
        "fallback_global_min_interval_score"
    )

best_gccp = (
    gccp_grid_df
    .loc[best_idx]
)

best_h_geo_gccp = float(
    best_gccp[
        "h_geo_km"
    ]
)

best_h_causal = float(
    best_gccp[
        "h_causal"
    ]
)

best_lambda_gccp = float(
    best_gccp[
        "lambda"
    ]
)

# ============================================================
# 16. Reconstruct selected result
# ============================================================

K_geo_best = (
    geo_kernels[
        best_h_geo_gccp
    ]
)

K_causal_best = (
    causal_kernels[
        best_h_causal
    ]
)

W_best = (
    best_lambda_gccp
    * K_geo_best
    +
    (1.0 - best_lambda_gccp)
    * K_causal_best
)

q_best = (
    weighted_q_matrix(
        W_best
    )
)

neff_best = (
    neff_matrix_stable(
        W_best
    )
)

lower_best = (
    pred_val
    - q_best
)

upper_best = (
    pred_val
    + q_best
)

covered_best = (
    (y_val >= lower_best)
    &
    (y_val <= upper_best)
)

width_best = (
    upper_best
    - lower_best
)

score_best = (
    interval_score(
        y_val,
        lower_best,
        upper_best,
        alpha=ALPHA
    )
)

# ============================================================
# 17. Output
# ============================================================

print("\n" + "=" * 80)
print("GCCP-B COVERAGE FEASIBILITY")
print("=" * 80)

print(
    "Coverage >= 0.90 feasible:",
    gccp_feasible
)

print(
    "Maximum validation coverage:",
    gccp_grid_df[
        "coverage"
    ].max()
)

print(
    "Selection rule:",
    gccp_selection_rule
)

print("\n" + "=" * 80)
print("FORMAL GCCP-B — SELECTED VALIDATION PARAMETER")
print("=" * 80)

print(
    f"h_geo (km)     = "
    f"{best_h_geo_gccp:.6f}"
)

print(
    f"h_causal       = "
    f"{best_h_causal:.6f}"
)

print(
    f"Lambda         = "
    f"{best_lambda_gccp:.6f}"
)

print(
    f"Coverage       = "
    f"{best_gccp['coverage']:.6f}"
)

print(
    f"Mean Width     = "
    f"{best_gccp['mean_width']:.6f}"
)

print(
    f"Interval Score = "
    f"{best_gccp['interval_score']:.6f}"
)

print(
    f"Median n_eff   = "
    f"{best_gccp['median_n_eff']:.6f}"
)

# ============================================================
# 18. Boundary diagnostics
# ============================================================

lambda_boundary = (
    np.isclose(
        best_lambda_gccp,
        LAMBDA_GRID_GCCP.min()
    )
    or
    np.isclose(
        best_lambda_gccp,
        LAMBDA_GRID_GCCP.max()
    )
)

h_geo_boundary = (
    np.isclose(
        best_h_geo_gccp,
        H_GEO_GRID_GCCP.min()
    )
    or
    np.isclose(
        best_h_geo_gccp,
        H_GEO_GRID_GCCP.max()
    )
)

h_causal_boundary = (
    np.isclose(
        best_h_causal,
        H_CAUSAL_GRID.min()
    )
    or
    np.isclose(
        best_h_causal,
        H_CAUSAL_GRID.max()
    )
)

print("\n" + "=" * 80)
print("GCCP-B BOUNDARY DIAGNOSTIC")
print("=" * 80)

print(
    "lambda boundary:",
    lambda_boundary
)

print(
    "h_geo boundary:",
    h_geo_boundary
)

print(
    "h_causal boundary:",
    h_causal_boundary
)

# ============================================================
# 19. Load Additive-EUC frozen result
# ============================================================

additive_df = pd.read_csv(
    ADDITIVE_PATH,
    encoding="utf-8-sig"
)

additive = (
    additive_df.iloc[0]
)

# ============================================================
# 20. Controlled causal comparison
# ============================================================

controlled = pd.DataFrame([
    {
        "method":
            "Additive-EUC",

        "lambda":
            additive[
                "lambda"
            ],

        "coverage":
            additive[
                "coverage"
            ],

        "mean_width":
            additive[
                "mean_width"
            ],

        "interval_score":
            additive[
                "interval_score"
            ],

        "median_n_eff":
            additive[
                "median_n_eff"
            ],
    },

    {
        "method":
            "GCCP-B",

        "lambda":
            best_lambda_gccp,

        "coverage":
            best_gccp[
                "coverage"
            ],

        "mean_width":
            best_gccp[
                "mean_width"
            ],

        "interval_score":
            best_gccp[
                "interval_score"
            ],

        "median_n_eff":
            best_gccp[
                "median_n_eff"
            ],
    }
])

print("\n" + "=" * 80)
print("CONTROLLED CAUSAL COMPARISON")
print("=" * 80)

display(
    controlled
)

# ============================================================
# 21. Relative changes
# ============================================================

add_cov = float(
    additive[
        "coverage"
    ]
)

add_width = float(
    additive[
        "mean_width"
    ]
)

add_score = float(
    additive[
        "interval_score"
    ]
)

print(
    "\nGCCP-B relative to Additive-EUC:"
)

print(
    "Coverage change (pp) =",
    100 * (
        best_gccp[
            "coverage"
        ]
        - add_cov
    )
)

print(
    "Width change (%)     =",
    100 * (
        best_gccp[
            "mean_width"
        ]
        / add_width
        - 1
    )
)

print(
    "Score change (%)     =",
    100 * (
        best_gccp[
            "interval_score"
        ]
        / add_score
        - 1
    )
)

# ============================================================
# 22. Save
# ============================================================

gccp_grid_df.to_csv(
    GCCP_OUTPUT_DIR
    / "pm25_gccp_b_validation_grid_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    **best_gccp.to_dict(),

    "coverage_constraint_feasible":
        gccp_feasible,

    "selection_rule":
        gccp_selection_rule,

    "geo_distance":
        "Haversine_km",

    "causal_distance":
        "GCCM_weighted_standardized_Euclidean",

    "fusion":
        "lambda*K_geo+(1-lambda)*K_causal",

    "gccm_source":
        "Train_only",

}]).to_csv(
    GCCP_OUTPUT_DIR
    / "pm25_gccp_b_validation_best_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame({
    "h_causal":
        H_CAUSAL_GRID
}).to_csv(
    GCCP_OUTPUT_DIR
    / "pm25_gccp_h_causal_grid_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame({
    "original_index":
        idx_val,

    "id":
        data.iloc[idx_val][
            "id"
        ].to_numpy(),

    "y_true":
        y_val,

    "y_pred":
        pred_val,

    "q_local":
        q_best,

    "lower":
        lower_best,

    "upper":
        upper_best,

    "width":
        width_best,

    "covered":
        covered_best.astype(int),

    "interval_score":
        score_best,

    "n_eff":
        neff_best,

}).to_csv(
    GCCP_OUTPUT_DIR
    / "pm25_gccp_b_validation_per_sample_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

controlled.to_csv(
    GCCP_OUTPUT_DIR
    / "pm25_gccp_b_vs_additive_validation_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ STEP 13B 完成")
print("✅ GCCP-B Validation 参数已冻结")
print("⚠️ Frozen Test y 仍未读取")

STEP 13B — PM2.5 GCCP-B / VALIDATION
Train       : 990
Calibration : 141
Validation  : 142

Conformal k: 128
tau_local: 0.9078014184397163

h_geo candidates: 25
lambda grid: [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]

FROZEN GCCP-B PARAMETER GRIDS
h_geo range: 1.5768620734001186 → 2331.6315954622364
h_causal range: 0.08949000085807299 → 5.695449958654519
h_causal candidates: 20
lambda: [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]

✅ Geographic and causal kernels precomputed

Grid combinations: 5500

GCCP-B COVERAGE FEASIBILITY
Coverage >= 0.90 feasible: False
Maximum validation coverage: 0.8591549295774648
Selection rule: fallback_global_min_interval_score

FORMAL GCCP-B — SELECTED VALIDATION PARAMETER
h_geo (km)     = 690.794537
h_causal       = 1.233069
Lambda         = 0.100000
Coverage       = 0.852113
Mean Width     = 17.277858
Interval Score = 31.307393
Median n_eff   = 68.074673

GCCP-B BOUNDARY DIAGNOSTIC
lambda boundary: False
h_geo boundary: False
h_causal boundary: False

,method,lambda,coverage,mean_width,interval_score,median_n_eff
0,Additive-EUC,0.2,0.823944,17.027888,31.899938,83.253205
1,GCCP-B,0.1,0.852113,17.277858,31.307393,68.074673



GCCP-B relative to Additive-EUC:
Coverage change (pp) = 2.8169014084507005
Width change (%)     = 1.4680027128715745
Score change (%)     = -1.8575142443874038

✅ STEP 13B 完成
✅ GCCP-B Validation 参数已冻结
⚠️ Frozen Test y 仍未读取


In [3]:
# ============================================================
# REPAIR — rebuild missing frozen parameter-grid files
#
# 原因：
# STEP 7 初版在 Coverage 不可行时提前 RuntimeError，
# 导致 h_geo grid 没有保存到磁盘。
#
# 这里严格按原冻结规则重建。
# 不使用 Validation y，更不使用 Frozen Test。
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 80)
print("REPAIR — FROZEN PARAMETER GRID FILES")
print("=" * 80)

ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

OUTPUT_DIR = (
    ROOT
    / "GCCP_PM2.5"
    / "outputs"
)

H_GEO_PATH = (
    OUTPUT_DIR
    / "pm25_formal_h_geo_grid_v1.csv"
)

LAMBDA_PATH = (
    OUTPUT_DIR
    / "pm25_formal_lambda_grid_v1.csv"
)

ADDITIVE_PATH = (
    OUTPUT_DIR
    / "pm25_formal_additive_euc_validation_best_v3.csv"
)

# ============================================================
# 1. D_geo 已由 STEP 13B 前半部分计算完成
# ============================================================

if "D_geo" not in globals():
    raise RuntimeError(
        "D_geo 不存在，请先重新运行 STEP 13B "
        "到 geographic distance 构造部分。"
    )

print("D_geo shape:", D_geo.shape)

assert D_geo.shape == (142, 141)

# ============================================================
# 2. 严格重建原 H_GEO_GRID
#
# 原冻结规则：
#
# h_min =
# max(1 km,
#     0.5 * 5th percentile nearest calibration distance)
#
# h_max =
# 90th percentile all Val-Calib geographic distances
#
# 25 geometric candidates
# ============================================================

nearest_calib_km = D_geo.min(
    axis=1
)

all_geo_dist = D_geo.ravel()

h_min = max(
    1.0,
    0.5
    * np.quantile(
        nearest_calib_km,
        0.05
    )
)

h_max = np.quantile(
    all_geo_dist,
    0.90
)

H_GEO_GRID_REBUILT = np.geomspace(
    h_min,
    h_max,
    25
)

print("\nRebuilt h_geo:")
print("h_min =", h_min)
print("h_max =", h_max)
print("n     =", len(H_GEO_GRID_REBUILT))

print(
    np.round(
        H_GEO_GRID_REBUILT,
        6
    )
)

# 我们之前 STEP 7 的已知审计结果大约为：
# h_min = 1.576862...
# h_max = 2331.631595...
#
# 这里只做 sanity check，不用于重新调参。

assert np.isclose(
    h_min,
    1.5768620734001186,
    rtol=1e-8,
    atol=1e-8
)

assert np.isclose(
    h_max,
    2331.6315954622364,
    rtol=1e-8,
    atol=1e-8
)

print(
    "\n✅ 重建的 h_geo 与 STEP 7 原冻结 grid 一致"
)

# ============================================================
# 3. 保存 h_geo grid
# ============================================================

pd.DataFrame({
    "h_geo_km":
        H_GEO_GRID_REBUILT
}).to_csv(
    H_GEO_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print(H_GEO_PATH)

# ============================================================
# 4. Lambda grid
#
# Additive-EUC / GCCP-B 固定：
# [0, 0.1, ..., 1.0]
# ============================================================

LAMBDA_GRID_REBUILT = np.linspace(
    0.0,
    1.0,
    11
)

if not LAMBDA_PATH.exists():

    pd.DataFrame({
        "lambda":
            LAMBDA_GRID_REBUILT
    }).to_csv(
        LAMBDA_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    print("\nLambda grid 文件原先不存在，已补建。")

else:

    lambda_check = pd.read_csv(
        LAMBDA_PATH,
        encoding="utf-8-sig"
    )["lambda"].to_numpy()

    assert np.allclose(
        lambda_check,
        LAMBDA_GRID_REBUILT
    )

    print(
        "\n✅ 已有 lambda grid 与冻结规则一致"
    )

print("Lambda path:")
print(LAMBDA_PATH)

# ============================================================
# 5. 检查 Additive-EUC 正式结果
# ============================================================

print("\nAdditive-EUC v3 exists:")
print(
    ADDITIVE_PATH.exists(),
    ADDITIVE_PATH
)

if not ADDITIVE_PATH.exists():
    raise FileNotFoundError(
        "找不到 Additive-EUC v3。"
        "需要检查 STEP 10C 是否保存成功。"
    )

# ============================================================
# 6. 最终检查
# ============================================================

assert H_GEO_PATH.exists()
assert LAMBDA_PATH.exists()
assert ADDITIVE_PATH.exists()

print("\n" + "=" * 80)
print("REPAIR COMPLETE")
print("=" * 80)

print("✅ h_geo grid 已恢复")
print("✅ lambda grid 已确认")
print("✅ Additive-EUC v3 已确认")
print("⚠️ Frozen Test 完全未使用")

REPAIR — FROZEN PARAMETER GRID FILES
D_geo shape: (142, 141)

Rebuilt h_geo:
h_min = 1.5768620734001186
h_max = 2331.6315954622364
n     = 25
[1.57686200e+00 2.13732900e+00 2.89700500e+00 3.92669300e+00
 5.32236600e+00 7.21410600e+00 9.77823000e+00 1.32537270e+01
 1.79645270e+01 2.43496960e+01 3.30043590e+01 4.47351670e+01
 6.06354800e+01 8.21872750e+01 1.11399268e+02 1.50994142e+02
 2.04662303e+02 2.77405851e+02 3.76004788e+02 5.09648951e+02
 6.90794537e+02 9.36325074e+02 1.26912504e+03 1.72021279e+03
 2.33163159e+03]

✅ 重建的 h_geo 与 STEP 7 原冻结 grid 一致

Saved:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_formal_h_geo_grid_v1.csv

✅ 已有 lambda grid 与冻结规则一致
Lambda path:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_formal_lambda_grid_v1.csv

Additive-EUC v3 exists:
True E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_formal_additive_euc_validation_best_v3.csv

REPAIR COMPLETE
✅ h_geo grid 已恢复
✅ lambda grid 已确认
✅ Additive-EUC v3 已确认
⚠️ F

In [5]:
# ============================================================
# STEP 13C — Freeze PM2.5 formal experiment before Test
#
# 目的：
# 把所有 Validation-selected 参数正式写入 manifest，
# 并生成 SHA256。
#
# 从这个 manifest 生成后：
# 不再允许重新选择参数。
# ============================================================

from pathlib import Path
import json
import hashlib
import pandas as pd
import numpy as np

print("=" * 80)
print("STEP 13C — FREEZE PM2.5 FORMAL EXPERIMENT")
print("=" * 80)

ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

OUTPUT_DIR = (
    ROOT
    / "GCCP_PM2.5"
    / "outputs"
)

GCCP_DIR = (
    OUTPUT_DIR
    / "gccp"
)

GCCM_DIR = (
    OUTPUT_DIR
    / "gccm"
)

# ============================================================
# 1. Required files
# ============================================================

paths = {
    "protocol":
        OUTPUT_DIR
        / "pm25_formal_protocol_v1.json",

    "cp":
        OUTPUT_DIR
        / "pm25_formal_cp_validation_summary_v1.csv",

    "geocp":
        OUTPUT_DIR
        / "pm25_formal_geocp_validation_best_v2.csv",

    "geosimcp_euc":
        OUTPUT_DIR
        / "pm25_formal_geosimcp_euc_validation_best_v1.csv",

    "geosimcp_mnd":
        OUTPUT_DIR
        / "pm25_formal_geosimcp_mnd_validation_best_v1.csv",

    "joint_hav_euc":
        OUTPUT_DIR
        / "pm25_formal_joint_hav_euc_validation_best_v1.csv",

    "additive_euc":
        OUTPUT_DIR
        / "pm25_formal_additive_euc_validation_best_v3.csv",

    "gccp_b":
        GCCP_DIR
        / "pm25_gccp_b_validation_best_v1.csv",

    "gccm_weights":
        GCCM_DIR
        / "pm25_gccm_causal_weights_v1.csv",
}

for name, p in paths.items():

    print(
        f"{name:<18s}: "
        f"{p.exists()}  {p}"
    )

    assert p.exists(), (
        f"Missing required file: {p}"
    )

# ============================================================
# 2. Helper
# ============================================================

def first_row(path):

    return (
        pd.read_csv(
            path,
            encoding="utf-8-sig"
        )
        .iloc[0]
    )


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()

# ============================================================
# 3. Load selected parameters
# ============================================================

cp = first_row(
    paths["cp"]
)

geocp = first_row(
    paths["geocp"]
)

euc = first_row(
    paths["geosimcp_euc"]
)

mnd = first_row(
    paths["geosimcp_mnd"]
)

joint = first_row(
    paths["joint_hav_euc"]
)

additive = first_row(
    paths["additive_euc"]
)

gccp = first_row(
    paths["gccp_b"]
)

weights_df = pd.read_csv(
    paths["gccm_weights"],
    encoding="utf-8-sig"
)

with open(
    paths["protocol"],
    "r",
    encoding="utf-8"
) as f:

    protocol = json.load(f)

# ============================================================
# 4. Extract frozen GCCM weights
# ============================================================

FEATURES = [
    "dem",
    "sp",
    "tp",
    "blh",
    "r",
    "aod_sat",
    "ndvi",
]

weight_lookup = (
    weights_df
    .set_index("feature")
)

gccm_weights = {}

for feature in FEATURES:

    gccm_weights[
        feature
    ] = {
        "causal_weight":
            float(
                weight_lookup.loc[
                    feature,
                    "causal_weight"
                ]
            ),

        "gamma":
            float(
                weight_lookup.loc[
                    feature,
                    "gamma"
                ]
            ),
    }

# ============================================================
# 5. Build formal manifest
# ============================================================

manifest = {

    "experiment":
        "PM2.5 formal frozen test v1",

    "split_sha256":
        protocol["split_sha256"],

    "alpha":
        0.1,

    "n": {
        "train": 990,
        "calibration": 141,
        "validation": 142,
        "frozen_test": 142,
    },

    "conformal": {
        "k": 128,
        "tau":
            128 / 141,
    },

    "predictor": {
        "model":
            "RandomForestRegressor",

        "n_estimators":
            300,

        "max_depth":
            None,

        "random_state":
            42,

        "selection":
            "Formal Train-only 3-fold CV",
    },

    "selection_rule": (
        "If Validation coverage >= 0.90 is feasible, "
        "choose minimum Interval Score among feasible settings; "
        "otherwise choose global minimum Validation Interval Score."
    ),

    "methods": {

        "CP": {
            "q":
                float(cp["q"]),
        },

        "GeoCP": {
            "h_geo_km":
                float(
                    geocp[
                        "bandwidth_km"
                    ]
                ),
        },

        "GeoSIMCP-EUC": {
            "bandwidth":
                float(
                    euc["bandwidth"]
                ),

            "lambda":
                float(
                    euc["lambda"]
                ),

            "geo_distance":
                "raw_lon_lat_Euclidean_degree",

            "feature_distance":
                "standardized_Euclidean",
        },

        "GeoSIMCP-MND": {
            "bandwidth":
                float(
                    mnd["bandwidth"]
                ),

            "lambda":
                float(
                    mnd["lambda"]
                ),

            "geo_distance":
                "raw_lon_lat_Euclidean_degree",

            "feature_distance":
                "MND_calibration_range",
        },

        "Joint-HAV-EUC": {
            "bandwidth":
                float(
                    joint["bandwidth"]
                ),

            "lambda":
                float(
                    joint["lambda"]
                ),

            "geo_distance":
                "Haversine_km",

            "feature_distance":
                "standardized_Euclidean",
        },

        "Additive-EUC": {
            "h_geo_km":
                float(
                    additive["h_geo_km"]
                ),

            "h_feat":
                float(
                    additive["h_feat"]
                ),

            "lambda":
                float(
                    additive["lambda"]
                ),

            "fusion":
                "lambda*K_geo+(1-lambda)*K_feat",
        },

        "GCCP-B": {
            "h_geo_km":
                float(
                    gccp["h_geo_km"]
                ),

            "h_causal":
                float(
                    gccp["h_causal"]
                ),

            "lambda":
                float(
                    gccp["lambda"]
                ),

            "fusion":
                "lambda*K_geo+(1-lambda)*K_causal",
        },
    },

    "gccm": {
        "data":
            "Formal Train only",

        "E":
            3,

        "tau":
            1,

        "b":
            4,

        "library_sizes":
            [10, 50, 100, 200, 400, 988],

        "direction_for_weight":
            "effect_xmap_cause",

        "weights":
            gccm_weights,

        "weights_file_sha256":
            sha256_file(
                paths[
                    "gccm_weights"
                ]
            ),
    },

    "test_policy": (
        "Frozen Test labels have not been used "
        "for model selection or UQ parameter selection. "
        "After this manifest is created, no parameter "
        "may be changed based on Frozen Test results."
    ),
}

# ============================================================
# 6. Save canonical JSON
# ============================================================

MANIFEST_PATH = (
    OUTPUT_DIR
    / "pm25_formal_frozen_test_manifest_v1.json"
)

manifest_text = json.dumps(
    manifest,
    ensure_ascii=False,
    indent=2,
    sort_keys=True
)

MANIFEST_PATH.write_text(
    manifest_text,
    encoding="utf-8"
)

# ============================================================
# 7. Manifest SHA256
# ============================================================

manifest_hash = hashlib.sha256(
    manifest_text.encode(
        "utf-8"
    )
).hexdigest()

HASH_PATH = (
    OUTPUT_DIR
    / "pm25_formal_frozen_test_manifest_v1_sha256.txt"
)

HASH_PATH.write_text(
    manifest_hash,
    encoding="utf-8"
)

# ============================================================
# 8. Compact parameter table
# ============================================================

freeze_table = pd.DataFrame([
    {
        "method":
            "CP",
        "p1":
            f"q={float(cp['q']):.6f}",
        "p2":
            "",
        "p3":
            "",
    },

    {
        "method":
            "GeoCP",
        "p1":
            f"h_geo={float(geocp['bandwidth_km']):.6f}",
        "p2":
            "",
        "p3":
            "",
    },

    {
        "method":
            "GeoSIMCP-EUC",
        "p1":
            f"h={float(euc['bandwidth']):.6f}",
        "p2":
            f"lambda={float(euc['lambda']):.6f}",
        "p3":
            "",
    },

    {
        "method":
            "GeoSIMCP-MND",
        "p1":
            f"h={float(mnd['bandwidth']):.6f}",
        "p2":
            f"lambda={float(mnd['lambda']):.6f}",
        "p3":
            "",
    },

    {
        "method":
            "Joint-HAV-EUC",
        "p1":
            f"h={float(joint['bandwidth']):.6f}",
        "p2":
            f"lambda={float(joint['lambda']):.6f}",
        "p3":
            "",
    },

    {
        "method":
            "Additive-EUC",
        "p1":
            f"h_geo={float(additive['h_geo_km']):.6f}",
        "p2":
            f"h_feat={float(additive['h_feat']):.6f}",
        "p3":
            f"lambda={float(additive['lambda']):.6f}",
    },

    {
        "method":
            "GCCP-B",
        "p1":
            f"h_geo={float(gccp['h_geo_km']):.6f}",
        "p2":
            f"h_causal={float(gccp['h_causal']):.6f}",
        "p3":
            f"lambda={float(gccp['lambda']):.6f}",
    },
])

print("\n" + "=" * 80)
print("FROZEN METHOD PARAMETERS")
print("=" * 80)

display(
    freeze_table
)

print("\nManifest:")
print(
    MANIFEST_PATH
)

print("\nManifest SHA256:")
print(
    manifest_hash
)

print("\n" + "=" * 80)
print("FORMAL EXPERIMENT FROZEN")
print("=" * 80)

print("✅ Split frozen")
print("✅ RF frozen")
print("✅ GCCM weights frozen")
print("✅ All UQ parameters frozen")
print("✅ Selection rule frozen")
print("⚠️ 从现在开始禁止根据 Test 结果修改任何上述参数")
print("⚠️ Frozen Test y 仍然尚未读取")

STEP 13C — FREEZE PM2.5 FORMAL EXPERIMENT
protocol          : True  E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_formal_protocol_v1.json
cp                : True  E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_formal_cp_validation_summary_v1.csv
geocp             : True  E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_formal_geocp_validation_best_v2.csv
geosimcp_euc      : True  E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_formal_geosimcp_euc_validation_best_v1.csv
geosimcp_mnd      : True  E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_formal_geosimcp_mnd_validation_best_v1.csv
joint_hav_euc     : True  E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_formal_joint_hav_euc_validation_best_v1.csv
additive_euc      : True  E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_formal_additive_euc_validation_best_v3.csv
gccp_b            : True  E:\GeoSIMCP_Reproduce\geoconforma

,method,p1,p2,p3
0,CP,q=9.133607,,
1,GeoCP,h_geo=690.794537,,
2,GeoSIMCP-EUC,h=1.647368,lambda=0.000000,
3,GeoSIMCP-MND,h=2.421053,lambda=0.157895,
4,Joint-HAV-EUC,h=1.638246,lambda=0.000000,
5,Additive-EUC,h_geo=150.994142,h_feat=1.638246,lambda=0.200000
6,GCCP-B,h_geo=690.794537,h_causal=1.233069,lambda=0.100000



Manifest:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_formal_frozen_test_manifest_v1.json

Manifest SHA256:
d2090944506e6ca37ead111dae5f508540668fe2ef826ddcd0c8e463f25d7553

FORMAL EXPERIMENT FROZEN
✅ Split frozen
✅ RF frozen
✅ GCCM weights frozen
✅ All UQ parameters frozen
✅ Selection rule frozen
⚠️ 从现在开始禁止根据 Test 结果修改任何上述参数
⚠️ Frozen Test y 仍然尚未读取


In [6]:
# ============================================================
# STEP 14 — PM2.5 FINAL FROZEN TEST
#
# IMPORTANT:
# - All parameters already frozen in STEP 13C.
# - No Test-based parameter selection.
# - Test labels are accessed only after ALL prediction
#   intervals have already been constructed.
# ============================================================

from pathlib import Path
import json
import hashlib
import math

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    pairwise_distances,
)
from sklearn.metrics.pairwise import haversine_distances


print("=" * 80)
print("STEP 14 — PM2.5 FINAL FROZEN TEST")
print("=" * 80)


# ============================================================
# 0. Paths
# ============================================================

ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

OUTPUT_DIR = (
    ROOT
    / "GCCP_PM2.5"
    / "outputs"
)

GCCP_OUTPUT_DIR = (
    OUTPUT_DIR
    / "gccp"
)

GCCM_OUTPUT_DIR = (
    OUTPUT_DIR
    / "gccm"
)

DATA_PATH = (
    ROOT
    / "GCCP_PM2.5"
    / "00_data_audit"
    / "china_sites_2018_05_PM2.5_all.csv"
)

SPLIT_PATH = (
    OUTPUT_DIR
    / "pm25_formal_split_assignment_v1.csv"
)

MANIFEST_PATH = (
    OUTPUT_DIR
    / "pm25_formal_frozen_test_manifest_v1.json"
)

MANIFEST_HASH_PATH = (
    OUTPUT_DIR
    / "pm25_formal_frozen_test_manifest_v1_sha256.txt"
)

WEIGHT_PATH = (
    GCCM_OUTPUT_DIR
    / "pm25_gccm_causal_weights_v1.csv"
)


# ============================================================
# 1. Verify manifest SHA256
# ============================================================

manifest_text = MANIFEST_PATH.read_text(
    encoding="utf-8"
)

manifest_hash_now = hashlib.sha256(
    manifest_text.encode("utf-8")
).hexdigest()

manifest_hash_frozen = (
    MANIFEST_HASH_PATH
    .read_text(encoding="utf-8")
    .strip()
)

print("\nManifest SHA256 frozen:")
print(manifest_hash_frozen)

print("\nManifest SHA256 current:")
print(manifest_hash_now)

assert (
    manifest_hash_now
    == manifest_hash_frozen
)

print("\n✅ Manifest SHA256 verified")


manifest = json.loads(
    manifest_text
)


# ============================================================
# 2. Load / verify split
# ============================================================

split_df = pd.read_csv(
    SPLIT_PATH,
    encoding="utf-8-sig"
)

hash_text = "\n".join(
    f"{row.original_index}|{row.id}|{row.split}"
    for row in split_df.itertuples()
)

split_hash_now = hashlib.sha256(
    hash_text.encode("utf-8")
).hexdigest()

assert (
    split_hash_now
    == manifest["split_sha256"]
)

print("✅ Split SHA256 verified")


# ============================================================
# 3. Recover EXACT formal permutation order
#
# This is important because RandomForest with fixed random_state
# can still depend on row order.
#
# Formal split was originally:
# rng = np.random.default_rng(42)
# perm = rng.permutation(1415)
# ============================================================

N = 1415
SEED = 42

rng = np.random.default_rng(
    SEED
)

perm = rng.permutation(
    np.arange(N)
)

idx_train = perm[:990]

idx_calib = perm[
    990:
    990 + 141
]

idx_val = perm[
    990 + 141:
    990 + 141 + 142
]

idx_test = perm[
    990 + 141 + 142:
]

assert len(idx_train) == 990
assert len(idx_calib) == 141
assert len(idx_val) == 142
assert len(idx_test) == 142

# Check sets against frozen split CSV
for name, idx in [
    ("train", idx_train),
    ("calibration", idx_calib),
    ("validation", idx_val),
    ("test", idx_test),
]:
    frozen_set = set(
        split_df.loc[
            split_df["split"] == name,
            "original_index"
        ].astype(int)
    )

    assert set(idx) == frozen_set

print("✅ Exact split permutation order recovered")


# ============================================================
# 4. Verify GCCM weight file
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


gccm_hash_now = sha256_file(
    WEIGHT_PATH
)

gccm_hash_frozen = (
    manifest[
        "gccm"
    ][
        "weights_file_sha256"
    ]
)

assert (
    gccm_hash_now
    == gccm_hash_frozen
)

print("✅ GCCM causal-weight file SHA256 verified")


# ============================================================
# 5. Data
# ============================================================

data = pd.read_csv(
    DATA_PATH,
    encoding="gbk"
)

data = data.rename(
    columns={
        "监测点编码": "id",
        "监测点名称": "name",
        "城市": "city",
        "经度": "lon",
        "纬度": "lat",
    }
)

assert len(data) == 1415


FEATURES = [
    "dem",
    "sp",
    "tp",
    "blh",
    "r",
    "aod_sat",
    "ndvi",
]

MODEL_FEATURES = (
    FEATURES
    + ["lon", "lat"]
)


# ============================================================
# 6. Formal RF — fixed, no tuning
# ============================================================

X_train = (
    data.iloc[idx_train]
    [MODEL_FEATURES]
)

X_calib = (
    data.iloc[idx_calib]
    [MODEL_FEATURES]
)

X_test = (
    data.iloc[idx_test]
    [MODEL_FEATURES]
)

y_train = (
    data.iloc[idx_train]
    ["PM2.5"]
    .to_numpy(dtype=float)
)

y_calib = (
    data.iloc[idx_calib]
    ["PM2.5"]
    .to_numpy(dtype=float)
)

formal_rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)

formal_rf.fit(
    X_train,
    y_train
)

pred_calib = formal_rf.predict(
    X_calib
)

pred_test = formal_rf.predict(
    X_test
)

calib_residual = np.abs(
    y_calib
    - pred_calib
)


# ============================================================
# 7. Conformal finite-sample convention
# ============================================================

ALPHA = 0.1

n_calib = len(
    calib_residual
)

k_local = math.ceil(
    (n_calib + 1)
    * (1 - ALPHA)
)

k_local = min(
    k_local,
    n_calib
)

TAU_LOCAL = (
    k_local
    / n_calib
)

assert k_local == 128


# Global CP q reconstructed
scores_sorted = np.sort(
    calib_residual
)

q_cp_recomputed = (
    scores_sorted[
        k_local - 1
    ]
)

q_cp_frozen = float(
    manifest[
        "methods"
    ][
        "CP"
    ][
        "q"
    ]
)

print("\nFrozen CP q     =", q_cp_frozen)
print("Recomputed CP q =", q_cp_recomputed)

assert np.isclose(
    q_cp_recomputed,
    q_cp_frozen,
    atol=1e-12
)

print("✅ Formal RF + calibration residual exactly reproduced")


# ============================================================
# 8. Helper functions
# ============================================================

resid_order = np.argsort(
    calib_residual
)

resid_sorted = (
    calib_residual[
        resid_order
    ]
)


def weighted_q_matrix(
    W,
    tau=TAU_LOCAL
):

    W = np.asarray(
        W,
        dtype=float
    )

    W_sorted = (
        W[
            :,
            resid_order
        ]
    )

    row_sum = (
        W_sorted.sum(
            axis=1,
            keepdims=True
        )
    )

    if np.any(
        (~np.isfinite(row_sum))
        |
        (row_sum <= 0)
    ):
        raise RuntimeError(
            "Invalid kernel weights"
        )

    cum = (
        np.cumsum(
            W_sorted,
            axis=1
        )
        / row_sum
    )

    idx = np.argmax(
        cum >= tau,
        axis=1
    )

    return (
        resid_sorted[
            idx
        ]
    )


def neff_matrix_stable(
    W
):

    W = np.asarray(
        W,
        dtype=float
    )

    row_max = np.max(
        W,
        axis=1,
        keepdims=True
    )

    if np.any(
        (~np.isfinite(row_max))
        |
        (row_max <= 0)
    ):
        raise RuntimeError(
            "Invalid weights in n_eff"
        )

    W_scaled = (
        W / row_max
    )

    return (
        W_scaled.sum(axis=1) ** 2
        /
        (W_scaled ** 2).sum(axis=1)
    )


def single_gaussian_kernel(
    D,
    h
):
    """
    Single-kernel case:
    row-wise shift is valid because multiplying all
    weights in one row by the same constant does not
    affect weighted quantile or n_eff.
    """

    log_w = (
        -0.5
        * (D / h) ** 2
    )

    log_w -= np.max(
        log_w,
        axis=1,
        keepdims=True
    )

    return np.exp(
        log_w
    )


def raw_gaussian_kernel(
    D,
    h
):
    """
    Used for additive fusion.
    Do NOT independently rescale component kernels.
    """

    log_w = (
        -0.5
        * (D / h) ** 2
    )

    log_w = np.clip(
        log_w,
        -700,
        0
    )

    return np.exp(
        log_w
    )


def interval_score(
    y_true,
    lower,
    upper,
    alpha=0.1
):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    lower = np.asarray(
        lower,
        dtype=float
    )

    upper = np.asarray(
        upper,
        dtype=float
    )

    width = (
        upper - lower
    )

    below = (
        (lower - y_true)
        * (y_true < lower)
    )

    above = (
        (y_true - upper)
        * (y_true > upper)
    )

    return (
        width
        + (2 / alpha) * below
        + (2 / alpha) * above
    )


# ============================================================
# 9. Test geographic distances
#
# NOTE:
# Test FEATURES/COORDINATES may be used now.
# Test LABELS are still not accessed.
# ============================================================

EARTH_RADIUS_KM = 6371.0088

coord_calib_rad = np.radians(
    data.iloc[idx_calib]
    [["lat", "lon"]]
    .to_numpy(dtype=float)
)

coord_test_rad = np.radians(
    data.iloc[idx_test]
    [["lat", "lon"]]
    .to_numpy(dtype=float)
)

D_geo_hav = (
    haversine_distances(
        coord_test_rad,
        coord_calib_rad
    )
    * EARTH_RADIUS_KM
)


# Raw lon/lat degree distance
coord_calib_degree = (
    data.iloc[idx_calib]
    [["lon", "lat"]]
    .to_numpy(dtype=float)
)

coord_test_degree = (
    data.iloc[idx_test]
    [["lon", "lat"]]
    .to_numpy(dtype=float)
)

D_geo_degree = pairwise_distances(
    coord_test_degree,
    coord_calib_degree,
    metric="euclidean"
)


# ============================================================
# 10. Feature-space distances
#
# Same Calibration-only scaler as Validation experiment.
# ============================================================

X_calib_feat_raw = (
    data.iloc[idx_calib]
    [FEATURES]
    .to_numpy(dtype=float)
)

X_test_feat_raw = (
    data.iloc[idx_test]
    [FEATURES]
    .to_numpy(dtype=float)
)

scaler = StandardScaler()

Z_calib = scaler.fit_transform(
    X_calib_feat_raw
)

Z_test = scaler.transform(
    X_test_feat_raw
)

D_euc = pairwise_distances(
    Z_test,
    Z_calib,
    metric="euclidean"
)

D_euc_for_geosim = (
    D_euc + 1e-8
)


# ============================================================
# 11. MND distance
# ============================================================

feature_ranges = np.ptp(
    X_calib_feat_raw,
    axis=0
)

feature_ranges_safe = (
    feature_ranges.copy()
)

feature_ranges_safe[
    np.isclose(
        feature_ranges_safe,
        0.0
    )
] = 1e-8

diff_mnd = np.abs(
    X_test_feat_raw[:, None, :]
    -
    X_calib_feat_raw[None, :, :]
)

D_mnd = np.max(
    diff_mnd
    /
    feature_ranges_safe[
        None,
        None,
        :
    ],
    axis=2
)


# ============================================================
# 12. Causal distance
# ============================================================

weights_df = pd.read_csv(
    WEIGHT_PATH,
    encoding="utf-8-sig"
)

gamma_lookup = (
    weights_df
    .set_index("feature")
    ["gamma"]
    .to_dict()
)

gamma = np.array(
    [
        gamma_lookup[f]
        for f in FEATURES
    ],
    dtype=float
)

assert np.isclose(
    gamma.mean(),
    1.0
)

diff_causal = (
    Z_test[:, None, :]
    -
    Z_calib[None, :, :]
)

D_causal = np.sqrt(
    np.sum(
        gamma[
            None,
            None,
            :
        ]
        * diff_causal**2,
        axis=2
    )
)


# ============================================================
# 13. Construct ALL frozen Test intervals
#
# Still NO y_test.
# ============================================================

method_payload = {}


# ------------------------------------------------------------
# CP
# ------------------------------------------------------------

q_cp = np.full(
    len(idx_test),
    q_cp_frozen,
    dtype=float
)

method_payload["CP"] = {
    "q": q_cp,
    "n_eff": np.full(
        len(idx_test),
        n_calib,
        dtype=float
    ),
}


# ------------------------------------------------------------
# GeoCP
# ------------------------------------------------------------

p = manifest[
    "methods"
][
    "GeoCP"
]

h_geo = float(
    p["h_geo_km"]
)

W = single_gaussian_kernel(
    D_geo_hav,
    h_geo
)

method_payload["GeoCP"] = {
    "q":
        weighted_q_matrix(W),

    "n_eff":
        neff_matrix_stable(W),
}


# ------------------------------------------------------------
# GeoSIMCP-EUC
# ------------------------------------------------------------

p = manifest[
    "methods"
][
    "GeoSIMCP-EUC"
]

h = float(
    p["bandwidth"]
)

lam = float(
    p["lambda"]
)

D_joint = np.sqrt(
    lam
    * D_geo_degree**2
    +
    (1 - lam)
    * D_euc_for_geosim**2
)

W = single_gaussian_kernel(
    D_joint,
    h
)

method_payload["GeoSIMCP-EUC"] = {
    "q":
        weighted_q_matrix(W),

    "n_eff":
        neff_matrix_stable(W),
}


# ------------------------------------------------------------
# GeoSIMCP-MND
# ------------------------------------------------------------

p = manifest[
    "methods"
][
    "GeoSIMCP-MND"
]

h = float(
    p["bandwidth"]
)

lam = float(
    p["lambda"]
)

D_joint = np.sqrt(
    lam
    * D_geo_degree**2
    +
    (1 - lam)
    * D_mnd**2
)

W = single_gaussian_kernel(
    D_joint,
    h
)

method_payload["GeoSIMCP-MND"] = {
    "q":
        weighted_q_matrix(W),

    "n_eff":
        neff_matrix_stable(W),
}


# ------------------------------------------------------------
# Joint-HAV-EUC
# ------------------------------------------------------------

p = manifest[
    "methods"
][
    "Joint-HAV-EUC"
]

h = float(
    p["bandwidth"]
)

lam = float(
    p["lambda"]
)

D_joint = np.sqrt(
    lam
    * D_geo_hav**2
    +
    (1 - lam)
    * D_euc**2
)

W = single_gaussian_kernel(
    D_joint,
    h
)

method_payload["Joint-HAV-EUC"] = {
    "q":
        weighted_q_matrix(W),

    "n_eff":
        neff_matrix_stable(W),
}


# ------------------------------------------------------------
# Additive-EUC
# ------------------------------------------------------------

p = manifest[
    "methods"
][
    "Additive-EUC"
]

h_geo = float(
    p["h_geo_km"]
)

h_feat = float(
    p["h_feat"]
)

lam = float(
    p["lambda"]
)

K_geo = raw_gaussian_kernel(
    D_geo_hav,
    h_geo
)

K_feat = raw_gaussian_kernel(
    D_euc,
    h_feat
)

W = (
    lam * K_geo
    +
    (1 - lam) * K_feat
)

method_payload["Additive-EUC"] = {
    "q":
        weighted_q_matrix(W),

    "n_eff":
        neff_matrix_stable(W),
}


# ------------------------------------------------------------
# GCCP-B
# ------------------------------------------------------------

p = manifest[
    "methods"
][
    "GCCP-B"
]

h_geo = float(
    p["h_geo_km"]
)

h_causal = float(
    p["h_causal"]
)

lam = float(
    p["lambda"]
)

K_geo = raw_gaussian_kernel(
    D_geo_hav,
    h_geo
)

K_causal = raw_gaussian_kernel(
    D_causal,
    h_causal
)

W = (
    lam * K_geo
    +
    (1 - lam) * K_causal
)

method_payload["GCCP-B"] = {
    "q":
        weighted_q_matrix(W),

    "n_eff":
        neff_matrix_stable(W),
}


# ============================================================
# 14. Verify all intervals constructed BEFORE Test labels
# ============================================================

METHOD_ORDER = [
    "CP",
    "GeoCP",
    "GeoSIMCP-EUC",
    "GeoSIMCP-MND",
    "Joint-HAV-EUC",
    "Additive-EUC",
    "GCCP-B",
]

for method in METHOD_ORDER:

    q = method_payload[
        method
    ][
        "q"
    ]

    neff = method_payload[
        method
    ][
        "n_eff"
    ]

    assert len(q) == 142
    assert len(neff) == 142
    assert np.all(np.isfinite(q))
    assert np.all(np.isfinite(neff))

    method_payload[
        method
    ][
        "lower"
    ] = (
        pred_test - q
    )

    method_payload[
        method
    ][
        "upper"
    ] = (
        pred_test + q
    )


print("\n" + "=" * 80)
print("ALL TEST INTERVALS CONSTRUCTED")
print("=" * 80)

print(
    "✅ 7 methods constructed "
    "without accessing Frozen Test labels"
)


# ============================================================
# 15. OPEN FROZEN TEST — ONE TIME
# ============================================================

print("\n" + "!" * 80)
print("OPENING FROZEN TEST LABELS — FINAL ONE-SHOT EVALUATION")
print("!" * 80)

y_test = (
    data.iloc[idx_test]
    ["PM2.5"]
    .to_numpy(dtype=float)
)

print(
    "Frozen Test labels accessed:",
    len(y_test)
)

assert len(y_test) == 142


# ============================================================
# 16. Predictor Test metrics
# ============================================================

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        pred_test
    )
)

rf_mae = mean_absolute_error(
    y_test,
    pred_test
)

rf_r2 = r2_score(
    y_test,
    pred_test
)

print("\n" + "=" * 80)
print("FORMAL RF — FROZEN TEST")
print("=" * 80)

print(
    f"RMSE = {rf_rmse:.6f}"
)

print(
    f"MAE  = {rf_mae:.6f}"
)

print(
    f"R²   = {rf_r2:.6f}"
)


# ============================================================
# 17. Evaluate ALL UQ methods
# ============================================================

summary_rows = []
per_sample_rows = []

for method in METHOD_ORDER:

    payload = method_payload[
        method
    ]

    q = payload["q"]
    lower = payload["lower"]
    upper = payload["upper"]
    neff = payload["n_eff"]

    width = (
        upper - lower
    )

    covered = (
        (y_test >= lower)
        &
        (y_test <= upper)
    )

    score = interval_score(
        y_test,
        lower,
        upper,
        alpha=ALPHA
    )

    summary_rows.append({
        "method":
            method,

        "n_test":
            len(y_test),

        "coverage":
            float(
                np.mean(
                    covered
                )
            ),

        "covered_n":
            int(
                covered.sum()
            ),

        "mean_half_width":
            float(
                np.mean(
                    q
                )
            ),

        "mean_width":
            float(
                np.mean(
                    width
                )
            ),

        "interval_score":
            float(
                np.mean(
                    score
                )
            ),

        "median_n_eff":
            float(
                np.median(
                    neff
                )
            ),

        "p10_n_eff":
            float(
                np.quantile(
                    neff,
                    0.10
                )
            ),

        "p90_n_eff":
            float(
                np.quantile(
                    neff,
                    0.90
                )
            ),
    })

    for j in range(
        len(idx_test)
    ):

        per_sample_rows.append({
            "method":
                method,

            "original_index":
                int(
                    idx_test[j]
                ),

            "id":
                str(
                    data.iloc[
                        idx_test[j]
                    ]["id"]
                ),

            "y_true":
                float(
                    y_test[j]
                ),

            "y_pred":
                float(
                    pred_test[j]
                ),

            "q":
                float(
                    q[j]
                ),

            "lower":
                float(
                    lower[j]
                ),

            "upper":
                float(
                    upper[j]
                ),

            "width":
                float(
                    width[j]
                ),

            "covered":
                int(
                    covered[j]
                ),

            "interval_score":
                float(
                    score[j]
                ),

            "n_eff":
                float(
                    neff[j]
                ),
        })


final_summary = pd.DataFrame(
    summary_rows
)

final_per_sample = pd.DataFrame(
    per_sample_rows
)


# ============================================================
# 18. Final frozen Test table
# ============================================================

print("\n" + "=" * 80)
print("PM2.5 FINAL FROZEN TEST SUMMARY")
print("=" * 80)

display(
    final_summary[
        [
            "method",
            "coverage",
            "covered_n",
            "mean_width",
            "interval_score",
            "median_n_eff",
        ]
    ]
)


# ============================================================
# 19. Controlled comparisons
# ============================================================

def get_row(method):

    return (
        final_summary
        .set_index("method")
        .loc[method]
    )


add = get_row(
    "Additive-EUC"
)

gccp = get_row(
    "GCCP-B"
)

joint = get_row(
    "Joint-HAV-EUC"
)

print("\n" + "=" * 80)
print("FINAL CONTROLLED COMPARISONS")
print("=" * 80)

print("\nGCCP-B relative to Additive-EUC:")

print(
    "Coverage change (pp) =",
    100 * (
        gccp["coverage"]
        - add["coverage"]
    )
)

print(
    "Width change (%)     =",
    100 * (
        gccp["mean_width"]
        / add["mean_width"]
        - 1
    )
)

print(
    "Score change (%)     =",
    100 * (
        gccp["interval_score"]
        / add["interval_score"]
        - 1
    )
)


print(
    "\nAdditive-EUC relative to Joint-HAV-EUC:"
)

print(
    "Coverage change (pp) =",
    100 * (
        add["coverage"]
        - joint["coverage"]
    )
)

print(
    "Width change (%)     =",
    100 * (
        add["mean_width"]
        / joint["mean_width"]
        - 1
    )
)

print(
    "Score change (%)     =",
    100 * (
        add["interval_score"]
        / joint["interval_score"]
        - 1
    )
)


# ============================================================
# 20. Save FINAL outputs
# ============================================================

SUMMARY_PATH = (
    OUTPUT_DIR
    / "pm25_formal_frozen_test_summary_v1.csv"
)

PER_SAMPLE_PATH = (
    OUTPUT_DIR
    / "pm25_formal_frozen_test_per_sample_v1.csv"
)

RF_PATH = (
    OUTPUT_DIR
    / "pm25_formal_frozen_test_rf_metrics_v1.csv"
)

COMPLETION_PATH = (
    OUTPUT_DIR
    / "pm25_formal_frozen_test_completion_v1.json"
)

final_summary.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

final_per_sample.to_csv(
    PER_SAMPLE_PATH,
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    "RMSE":
        rf_rmse,

    "MAE":
        rf_mae,

    "R2":
        rf_r2,

    "manifest_sha256":
        manifest_hash_now,
}]).to_csv(
    RF_PATH,
    index=False,
    encoding="utf-8-sig"
)


completion = {
    "status":
        "FINAL_FROZEN_TEST_COMPLETED",

    "manifest_sha256":
        manifest_hash_now,

    "split_sha256":
        split_hash_now,

    "n_test":
        142,

    "methods":
        METHOD_ORDER,

    "policy":
        (
            "All Test intervals were constructed "
            "using frozen parameters before Test "
            "labels were accessed."
        ),
}

COMPLETION_PATH.write_text(
    json.dumps(
        completion,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print("\nSaved:")
print(SUMMARY_PATH)
print(PER_SAMPLE_PATH)
print(RF_PATH)
print(COMPLETION_PATH)

print("\n" + "=" * 80)
print("PM2.5 FORMAL EXPERIMENT COMPLETE")
print("=" * 80)

print("✅ Frozen Test evaluated exactly once")
print("✅ All seven methods evaluated together")
print("✅ No Test-based parameter selection")
print("⛔ Do NOT retune after seeing these results")

STEP 14 — PM2.5 FINAL FROZEN TEST

Manifest SHA256 frozen:
d2090944506e6ca37ead111dae5f508540668fe2ef826ddcd0c8e463f25d7553

Manifest SHA256 current:
d2090944506e6ca37ead111dae5f508540668fe2ef826ddcd0c8e463f25d7553

✅ Manifest SHA256 verified
✅ Split SHA256 verified
✅ Exact split permutation order recovered
✅ GCCM causal-weight file SHA256 verified

Frozen CP q     = 9.133607116712255
Recomputed CP q = 9.133607116712255
✅ Formal RF + calibration residual exactly reproduced

ALL TEST INTERVALS CONSTRUCTED
✅ 7 methods constructed without accessing Frozen Test labels

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
OPENING FROZEN TEST LABELS — FINAL ONE-SHOT EVALUATION
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Frozen Test labels accessed: 142

FORMAL RF — FROZEN TEST
RMSE = 5.967174
MAE  = 4.315829
R²   = 0.757581

PM2.5 FINAL FROZEN TEST SUMMARY


,method,coverage,covered_n,mean_width,interval_score,median_n_eff
0,CP,0.894366,127,18.267214,27.221021,141.000000
1,GeoCP,0.873239,124,17.553033,25.593470,81.720548
2,GeoSIMCP-EUC,0.866197,123,16.959461,26.972864,83.059654
3,GeoSIMCP-MND,0.866197,123,17.272900,25.658859,69.927360
4,Joint-HAV-EUC,0.866197,123,16.960444,26.973847,82.512900
5,Additive-EUC,0.859155,122,17.021409,26.930548,80.072098
6,GCCP-B,0.866197,123,17.121002,25.519676,64.046931



FINAL CONTROLLED COMPARISONS

GCCP-B relative to Additive-EUC:
Coverage change (pp) = 0.7042253521126751
Width change (%)     = 0.5851034476298533
Score change (%)     = -5.238927743980948

Additive-EUC relative to Joint-HAV-EUC:
Coverage change (pp) = -0.7042253521126751
Width change (%)     = 0.3594543854444021
Score change (%)     = -0.16052213840285212

Saved:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_formal_frozen_test_summary_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_formal_frozen_test_per_sample_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_formal_frozen_test_rf_metrics_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\pm25_formal_frozen_test_completion_v1.json

PM2.5 FORMAL EXPERIMENT COMPLETE
✅ Frozen Test evaluated exactly once
✅ All seven methods evaluated together
✅ No Test-based parameter selection
⛔ Do NOT retune after seeing these results


In [7]:
# ============================================================
# STEP 15A — Frozen Test post-hoc diagnostic
#
# NO tuning.
# NO parameter changes.
#
# Purpose:
# Understand WHY GCCP-B has much lower Interval Score
# than Additive-EUC despite similar width / coverage.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 80)
print("STEP 15A — TEST SCORE DECOMPOSITION")
print("=" * 80)

ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

OUTPUT_DIR = (
    ROOT
    / "GCCP_PM2.5"
    / "outputs"
)

PER_SAMPLE_PATH = (
    OUTPUT_DIR
    / "pm25_formal_frozen_test_per_sample_v1.csv"
)

df = pd.read_csv(
    PER_SAMPLE_PATH,
    encoding="utf-8-sig"
)

# ============================================================
# 1. Extract controlled pair
# ============================================================

add = (
    df[
        df["method"]
        == "Additive-EUC"
    ]
    .copy()
)

gccp = (
    df[
        df["method"]
        == "GCCP-B"
    ]
    .copy()
)

pair = add.merge(
    gccp,
    on=[
        "original_index",
        "id"
    ],
    suffixes=(
        "_add",
        "_gccp"
    )
)

assert len(pair) == 142

# ============================================================
# 2. Interval Score decomposition
#
# IS = width + miss penalty
# ============================================================

pair[
    "penalty_add"
] = (
    pair["interval_score_add"]
    - pair["width_add"]
)

pair[
    "penalty_gccp"
] = (
    pair["interval_score_gccp"]
    - pair["width_gccp"]
)

pair[
    "delta_width"
] = (
    pair["width_gccp"]
    - pair["width_add"]
)

pair[
    "delta_penalty"
] = (
    pair["penalty_gccp"]
    - pair["penalty_add"]
)

pair[
    "delta_score"
] = (
    pair["interval_score_gccp"]
    - pair["interval_score_add"]
)

print("\n" + "=" * 80)
print("MEAN SCORE DECOMPOSITION")
print("=" * 80)

print(
    "Additive mean width   =",
    pair["width_add"].mean()
)

print(
    "GCCP-B mean width     =",
    pair["width_gccp"].mean()
)

print(
    "\nAdditive miss penalty =",
    pair["penalty_add"].mean()
)

print(
    "GCCP-B miss penalty   =",
    pair["penalty_gccp"].mean()
)

print(
    "\nΔ Width               =",
    pair["delta_width"].mean()
)

print(
    "Δ Miss penalty        =",
    pair["delta_penalty"].mean()
)

print(
    "Δ Interval Score      =",
    pair["delta_score"].mean()
)

# ============================================================
# 3. Coverage transitions
# ============================================================

both_cover = np.sum(
    (pair["covered_add"] == 1)
    &
    (pair["covered_gccp"] == 1)
)

both_miss = np.sum(
    (pair["covered_add"] == 0)
    &
    (pair["covered_gccp"] == 0)
)

gccp_only = np.sum(
    (pair["covered_add"] == 0)
    &
    (pair["covered_gccp"] == 1)
)

add_only = np.sum(
    (pair["covered_add"] == 1)
    &
    (pair["covered_gccp"] == 0)
)

print("\n" + "=" * 80)
print("COVERAGE TRANSITION")
print("=" * 80)

print(
    "Both cover      :",
    both_cover
)

print(
    "Both miss       :",
    both_miss
)

print(
    "GCCP-B only     :",
    gccp_only
)

print(
    "Additive only   :",
    add_only
)

print(
    "Net GCCP gain   :",
    gccp_only
    - add_only
)

# ============================================================
# 4. Is score gain widespread or driven by few samples?
# ============================================================

print("\n" + "=" * 80)
print("PER-SAMPLE SCORE CHANGE")
print("=" * 80)

print(
    "Fraction GCCP score better:",
    np.mean(
        pair[
            "delta_score"
        ] < 0
    )
)

print(
    "Median ΔScore:",
    np.median(
        pair[
            "delta_score"
        ]
    )
)

print(
    "p10 ΔScore:",
    np.quantile(
        pair[
            "delta_score"
        ],
        0.10
    )
)

print(
    "p90 ΔScore:",
    np.quantile(
        pair[
            "delta_score"
        ],
        0.90
    )
)

# ============================================================
# 5. Paired descriptive bootstrap
#
# This is a robustness diagnostic, NOT parameter selection.
# ============================================================

rng = np.random.default_rng(
    42
)

B = 10000
n = len(pair)

boot_width = np.empty(B)
boot_score = np.empty(B)
boot_cov = np.empty(B)

covered_diff = (
    pair["covered_gccp"].to_numpy()
    -
    pair["covered_add"].to_numpy()
)

width_diff = (
    pair["delta_width"].to_numpy()
)

score_diff = (
    pair["delta_score"].to_numpy()
)

for b in range(B):

    ii = rng.integers(
        0,
        n,
        size=n
    )

    boot_width[b] = (
        width_diff[ii].mean()
    )

    boot_score[b] = (
        score_diff[ii].mean()
    )

    boot_cov[b] = (
        covered_diff[ii].mean()
        * 100
    )


def ci95(x):
    return (
        np.quantile(
            x,
            0.025
        ),
        np.quantile(
            x,
            0.975
        )
    )


print("\n" + "=" * 80)
print("PAIRED BOOTSTRAP 95% CI")
print("=" * 80)

print(
    "Δ Width:",
    width_diff.mean(),
    "CI:",
    ci95(
        boot_width
    )
)

print(
    "Δ Score:",
    score_diff.mean(),
    "CI:",
    ci95(
        boot_score
    )
)

print(
    "Δ Coverage (pp):",
    covered_diff.mean() * 100,
    "CI:",
    ci95(
        boot_cov
    )
)

# ============================================================
# 6. Save
# ============================================================

pair.to_csv(
    OUTPUT_DIR
    / "pm25_gccp_vs_additive_test_decomposition_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ STEP 15A 完成")
print("⛔ This analysis does NOT permit any retuning")

STEP 15A — TEST SCORE DECOMPOSITION

MEAN SCORE DECOMPOSITION
Additive mean width   = 17.021408707734427
GCCP-B mean width     = 17.12100155691855

Additive miss penalty = 9.909138813248626
GCCP-B miss penalty   = 8.398674038381753

Δ Width               = 0.09959284918412019
Δ Miss penalty        = -1.5104647748668747
Δ Interval Score      = -1.4108719256827544

COVERAGE TRANSITION
Both cover      : 122
Both miss       : 19
GCCP-B only     : 1
Additive only   : 0
Net GCCP gain   : 1

PER-SAMPLE SCORE CHANGE
Fraction GCCP score better: 0.33098591549295775
Median ΔScore: 0.0
p10 ΔScore: -2.307215821978817
p90 ΔScore: 1.0443926453650345

PAIRED BOOTSTRAP 95% CI
Δ Width: 0.09959284918412019 CI: (np.float64(-0.12393924700729454), np.float64(0.3365867908071968))
Δ Score: -1.4108719256827544 CI: (np.float64(-2.8007328428475766), np.float64(-0.2925205530705547))
Δ Coverage (pp): 0.7042253521126761 CI: (np.float64(0.0), np.float64(2.112676056338028))

✅ STEP 15A 完成
⛔ This analysis does NOT per

In [8]:
# ============================================================
# STEP 15B — Score-gain concentration diagnostic
#
# Post-hoc diagnostic only.
# NO tuning / NO parameter changes.
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("STEP 15B — SCORE GAIN CONCENTRATION")
print("=" * 80)

# pair 已在 STEP 15A 中生成
assert len(pair) == 142

# GCCP improvement:
# positive value = GCCP-B has LOWER score
pair["score_gain_gccp"] = (
    pair["interval_score_add"]
    - pair["interval_score_gccp"]
)

pair["penalty_gain_gccp"] = (
    pair["penalty_add"]
    - pair["penalty_gccp"]
)

# ============================================================
# 1. Overall gain
# ============================================================

total_gain = pair[
    "score_gain_gccp"
].sum()

mean_gain = pair[
    "score_gain_gccp"
].mean()

print("Total score gain =", total_gain)
print("Mean score gain  =", mean_gain)

# ============================================================
# 2. Largest contributors
# ============================================================

ranked = (
    pair.sort_values(
        "score_gain_gccp",
        ascending=False
    )
    .copy()
)

ranked["rank"] = np.arange(
    1,
    len(ranked) + 1
)

ranked["gain_fraction"] = (
    ranked["score_gain_gccp"]
    / total_gain
)

ranked["cumulative_gain_fraction"] = (
    ranked["score_gain_gccp"]
    .cumsum()
    / total_gain
)

print("\n" + "=" * 80)
print("TOP SCORE-GAIN SAMPLES")
print("=" * 80)

display(
    ranked[
        [
            "rank",
            "id",
            "covered_add",
            "covered_gccp",
            "width_add",
            "width_gccp",
            "penalty_add",
            "penalty_gccp",
            "score_gain_gccp",
            "gain_fraction",
            "cumulative_gain_fraction",
        ]
    ].head(15)
)

# ============================================================
# 3. Concentration
# ============================================================

positive = ranked[
    ranked["score_gain_gccp"] > 0
].copy()

positive_total = positive[
    "score_gain_gccp"
].sum()

print("\n" + "=" * 80)
print("GAIN CONCENTRATION")
print("=" * 80)

print(
    "Samples with GCCP score improvement:",
    len(positive),
    "/",
    len(pair)
)

for k in [1, 3, 5, 10, 20]:

    k = min(
        k,
        len(positive)
    )

    frac = (
        positive[
            "score_gain_gccp"
        ]
        .head(k)
        .sum()
        / positive_total
    )

    print(
        f"Top {k:2d} share of positive gain = "
        f"{frac:.4f}"
    )

# ============================================================
# 4. Common-miss subset
# ============================================================

common_miss = pair[
    (pair["covered_add"] == 0)
    &
    (pair["covered_gccp"] == 0)
].copy()

print("\n" + "=" * 80)
print("COMMON-MISS SUBSET")
print("=" * 80)

print("n =", len(common_miss))

print(
    "Additive mean penalty =",
    common_miss[
        "penalty_add"
    ].mean()
)

print(
    "GCCP-B mean penalty   =",
    common_miss[
        "penalty_gccp"
    ].mean()
)

print(
    "Mean penalty change   =",
    (
        common_miss[
            "penalty_gccp"
        ]
        -
        common_miss[
            "penalty_add"
        ]
    ).mean()
)

print(
    "Fraction common misses with smaller GCCP penalty =",
    np.mean(
        common_miss[
            "penalty_gccp"
        ]
        <
        common_miss[
            "penalty_add"
        ]
    )
)

# ============================================================
# 5. Leave-one-out mean ΔScore
#
# Test whether one point alone drives result.
# Negative = GCCP better.
# ============================================================

delta = pair[
    "delta_score"
].to_numpy(dtype=float)

loo_mean = np.array([
    np.delete(
        delta,
        i
    ).mean()
    for i in range(
        len(delta)
    )
])

print("\n" + "=" * 80)
print("LEAVE-ONE-OUT ΔSCORE")
print("=" * 80)

print(
    "Original mean ΔScore =",
    delta.mean()
)

print(
    "LOO minimum          =",
    loo_mean.min()
)

print(
    "LOO maximum          =",
    loo_mean.max()
)

print(
    "All LOO means < 0    =",
    np.all(
        loo_mean < 0
    )
)

# ============================================================
# 6. Trimmed sensitivity
# ============================================================

# Remove samples with largest absolute ΔScore
abs_order = np.argsort(
    np.abs(delta)
)[::-1]

trim_rows = []

for remove_n in [
    0,
    1,
    3,
    5,
    10
]:

    keep = np.ones(
        len(delta),
        dtype=bool
    )

    if remove_n > 0:
        keep[
            abs_order[
                :remove_n
            ]
        ] = False

    trim_rows.append({
        "removed_largest_abs_delta":
            remove_n,

        "remaining_n":
            keep.sum(),

        "mean_delta_score":
            delta[
                keep
            ].mean(),
    })

trim_df = pd.DataFrame(
    trim_rows
)

print("\n" + "=" * 80)
print("TRIMMED SENSITIVITY")
print("=" * 80)

display(
    trim_df
)

# ============================================================
# 7. Save
# ============================================================

ranked.to_csv(
    OUTPUT_DIR
    / "pm25_gccp_score_gain_concentration_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

trim_df.to_csv(
    OUTPUT_DIR
    / "pm25_gccp_score_trimmed_sensitivity_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ STEP 15B 完成")
print("⛔ PM2.5 parameters remain permanently frozen")

STEP 15B — SCORE GAIN CONCENTRATION
Total score gain = 200.34381344695112
Mean score gain  = 1.4108719256827544

TOP SCORE-GAIN SAMPLES


,rank,id,covered_add,covered_gccp,width_add,width_gccp,penalty_add,penalty_gccp,score_gain_gccp,gain_fraction,cumulative_gain_fraction
24,1,2691A,0,0,18.870457,24.131864,53.241101,0.627031,47.352663,0.236357,0.236357
106,2,2696A,0,0,18.870457,24.095964,210.698274,158.443210,47.029557,0.234744,0.471101
135,3,1956A,0,0,24.095964,29.212146,206.042757,154.880935,46.045639,0.229833,0.700934
31,4,1124A,0,0,13.896762,15.967643,74.467607,53.758796,18.637930,0.093030,0.793964
35,5,2340A,0,1,14.219006,17.012036,21.274263,0.000000,18.481233,0.092248,0.886212
27,6,2526A,0,0,15.967643,17.907865,56.957013,37.554799,17.461993,0.087160,0.973372
77,7,1818A,0,0,15.967643,17.012036,15.927547,5.483620,9.399534,0.046917,1.020289
44,8,1938A,0,0,17.012036,17.907865,65.607880,56.649592,8.062459,0.040243,1.060532
7,9,1042A,0,0,17.012036,17.907865,187.096951,178.138662,8.062459,0.040243,1.100775
75,10,1451A,1,1,23.046679,19.455285,0.000000,0.000000,3.591394,0.017926,1.118701



GAIN CONCENTRATION
Samples with GCCP score improvement: 47 / 142
Top  1 share of positive gain = 0.1758
Top  3 share of positive gain = 0.5214
Top  5 share of positive gain = 0.6592
Top 10 share of positive gain = 0.8321
Top 20 share of positive gain = 0.9210

COMMON-MISS SUBSET
n = 19
Additive mean penalty = 72.93807625029075
GCCP-B mean penalty   = 62.769037550010985
Mean penalty change   = -10.169038700279758
Fraction common misses with smaller GCCP penalty = 0.5263157894736842

LEAVE-ONE-OUT ΔSCORE
Original mean ΔScore = -1.4108719256827544
LOO minimum          = -1.5447220335120198
LOO maximum          = -1.0850436195563944
All LOO means < 0    = True

TRIMMED SENSITIVITY


,removed_largest_abs_delta,remaining_n,mean_delta_score
0,0,142,-1.410872
1,1,141,-1.085044
2,3,139,-0.431050
3,5,137,-0.166400
4,10,132,-0.143114



✅ STEP 15B 完成
⛔ PM2.5 parameters remain permanently frozen
